# Bearing Fault Classification — Full Pipeline
Run each cell in order. The pipeline takes ~30–60 min on a Colab CPU session.
Switch to a T4 GPU runtime to cut training time by ~5×.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip -q install torch pandas numpy scikit-learn xgboost imbalanced-learn \
               shap lime matplotlib seaborn streamlit optuna tqdm joblib

In [ ]:
# ── 2. Clone / upload project files ─────────────────────────────────────────
# Option A: upload this project's ZIP
# from google.colab import files
# files.upload()  # upload bearing_fault_project.zip
# !unzip -q bearing_fault_project.zip

# Option B: clone from GitHub (replace URL)
# !git clone https://github.com/YOUR_USERNAME/bearing-fault-classification
# %cd bearing-fault-classification

# For now just verify the src/ folder is present
import os
assert os.path.exists('src/config.py'), 'Upload the project files first'
print('Project files found.')

In [ ]:
# ── 3. Upload raw dataset ────────────────────────────────────────────────────
from google.colab import files
import shutil, os

print('Upload featuretime48k2048load_1.csv')
uploaded = files.upload()
fname    = list(uploaded.keys())[0]
os.makedirs('data', exist_ok=True)
shutil.move(fname, 'data/featuretime48k2048load_1.csv')
print(f'Dataset saved to data/{fname}')

In [ ]:
# ── 4. Preprocessing ─────────────────────────────────────────────────────────
from src.preprocessing import preprocess_pipeline
preprocess_pipeline()

In [ ]:
# ── 5. Train baseline models ─────────────────────────────────────────────────
from src.train_baselines import train_baselines
baseline_metrics = train_baselines()

In [ ]:
# ── 6. Train deep learning models ───────────────────────────────────────────
from src.train_deep import train_deep_models
deep_metrics = train_deep_models()

In [ ]:
# ── 7. Train meta-learning models ───────────────────────────────────────────
from src.train_meta import train_meta_models
meta_metrics = train_meta_models()

In [ ]:
# ── 8. Continual learning (FBCL) ────────────────────────────────────────────
from src.train_continual import train_fbcl
fbcl_model, fbcl_metrics = train_fbcl()

In [ ]:
# ── 9. Evaluate all models and generate comparison table ────────────────────
import json, os
import pandas as pd
from src.config import OUTPUT_DIR
from src.evaluate import compare_models

all_metrics = {}
for fname in os.listdir(OUTPUT_DIR):
    if fname.endswith('_metrics.json'):
        name = fname.replace('_metrics.json', '')
        with open(os.path.join(OUTPUT_DIR, fname)) as f:
            all_metrics[name] = json.load(f)

compare_models(all_metrics)

In [ ]:
# ── 10. Noise robustness test ───────────────────────────────────────────────
import numpy as np
import joblib
from src.preprocessing import load_preprocessed
from src.evaluate import noise_robustness_test, plot_noise_robustness
from src.config import MODEL_DIR

_, _, _, _, X_test, y_test, _, _ = load_preprocessed()
rf  = joblib.load(os.path.join(MODEL_DIR, 'rf_model.pkl'))
xgb = joblib.load(os.path.join(MODEL_DIR, 'xgb_model.pkl'))

noise_results = {
    'RF':  noise_robustness_test(rf.predict,  X_test, y_test),
    'XGB': noise_robustness_test(xgb.predict, X_test, y_test),
}
plot_noise_robustness(noise_results)
print(noise_results)

In [ ]:
# ── 11. SHAP / LIME interpretability ────────────────────────────────────────
import joblib, torch
from src.preprocessing import load_preprocessed
from src.models import FaultTransformer
from src.interpret import run_all_interpretability
from src.config import MODEL_DIR

X_train, _, _, _, X_test, y_test, _, _ = load_preprocessed()

sklearn_models = {
    'RF':  joblib.load(os.path.join(MODEL_DIR, 'rf_model.pkl')),
    'XGB': joblib.load(os.path.join(MODEL_DIR, 'xgb_model.pkl')),
    'GBM': joblib.load(os.path.join(MODEL_DIR, 'gbm_model.pkl')),
}

n_feat   = X_train.shape[1]
trans    = FaultTransformer(n_feat)
ckpt     = torch.load(os.path.join(MODEL_DIR, 'transformer_model.pth'), map_location='cpu')
trans.load_state_dict(ckpt['state_dict'])
trans.eval()
torch_models = {'Transformer': trans}

run_all_interpretability(sklearn_models, torch_models, X_train, X_test, y_test)

In [ ]:
# ── 12. Launch Streamlit dashboard (Colab tunnel) ───────────────────────────
!pip -q install pyngrok
from pyngrok import ngrok
import subprocess, time

proc = subprocess.Popen(['streamlit', 'run', 'dashboard/app.py',
                         '--server.port', '8501',
                         '--server.headless', 'true'])
time.sleep(4)
tunnel = ngrok.connect(8501)
print('Dashboard URL:', tunnel.public_url)

In [ ]:
# ── 13. Download all outputs ─────────────────────────────────────────────────
import zipfile, os
from google.colab import files
from src.config import OUTPUT_DIR, MODEL_DIR

with zipfile.ZipFile('results.zip', 'w') as z:
    for folder in [OUTPUT_DIR, MODEL_DIR]:
        for fname in os.listdir(folder):
            z.write(os.path.join(folder, fname), os.path.join(folder.split('/')[-1], fname))

files.download('results.zip')
print('results.zip downloaded')